# Log Data Analysis Codebook

This notebook performs all of the analysis reported in the results section of the paper. It is structured into the following main sections:
1. Setup and loading data
2. Calculating reliability of the codebook
3. Generating the codebook
4. Summary statistics
5. Other statistics reported in the paper

## 1. Setup and loading data
We begin by importing necessary modules and loading the data. This uses a Docker container that contains the log data inside of it. This is deliberately not directly replicable as we did not have access to publish students' log data.

In [1]:
from analysis.loading_services.get_data import *
from analysis.core_filter_functions import *
from analysis.filter_functions import *
from analysis.stats_functions import *
from analysis.stats_helper_functions import *
#from summary_stats_functions import *

coded_program_logs: DataFrame = get_data("coded_program_logs") #TODO: Does this need filtering?
codes: list[str] = get_data("codes")["code_name"].values.tolist()
participants: DataFrame = filter_null_data(get_data("participants"), "participants") #TODO: Currently this returns 90 participants; needs to be exactly 73
program_logs: DataFrame = clear_unattempted_exercises(filter_null_data(get_data("program_logs"),"program_logs"))
participants_list: list[str] = participants['participantid'].values.tolist()

dataframes = {
    "coded_program_logs": coded_program_logs,
    "codes": codes,
    "participants": participants,
    "program_logs": program_logs
}

## Reliability of codebook

## Generating codebook
Now the codebook that is reported in Appendix of the paper is generated. See `results/codebook_frequency.csv` for the resultant codebook and frequencies.

In [3]:
from analysis.codebook_analysis import *

save_code_counts()
print("Codebook saved to results/codebook_frequency.csv\n")

print(f"Number of codes: {len(flatten_codebook_tree(create_codebook_with_frequencies()))}\n")
most_frequent_codes = get_most_frequent_codes(n=20)
print(f"Most frequent codes:")
for label, count in most_frequent_codes:
    print(f"- {label}: {count} ({count / 322 * 100:.2f}%)")

most_frequent_bottom_level_codes = get_most_frequent_bottom_level_codes(n=20)
print("\nMost frequently bottom level codes:")
for label, count in most_frequent_bottom_level_codes:
    print(f"- {label}: {count} ({count / 322 * 100:.2f}%)")

Codebook saved to results/codebook_frequency.csv

Number of codes: 77

Most frequent codes:
- First move: 322 (100.00%)
- Resolved error(s): 268 (83.23%)
- Positive debugging indicators: 259 (80.43%)
- First change on line referred to in error message: 257 (79.81%)
- Introduced additional error(s): 245 (76.09%)
- Miscellaneous behaviour: 243 (75.47%)
- Ran code before making changes: 212 (65.84%)
- Inconsequential changes: 209 (64.91%)
- Resolved syntax error(s): 206 (63.98%)
- Repeated runs: 205 (63.66%)
- Introduced additional syntax error(s): 184 (57.14%)
- Resolved logical error(s): 170 (52.80%)
- Correctly resolved logical error: 164 (50.93%)
- Reverted previous changes: 159 (49.38%)
- Incorrect syntactical changes to existing statements: 158 (49.07%)
- Changes to existing statements in the program: 152 (47.20%)
- Correctly resolved syntax error: 148 (45.96%)
- Inconsequential changes to whitespace of program: 136 (42.24%)
- Repeated unsuccessful runs with no changes: 135 (41.93%)

## Summary statistics

Some of these were used to generate Table X in the paper.

In [34]:
print(f"Number of attempted exercises: {get_number_attempted_exercises(filtered_ids={}, **dataframes)}")
print(f"Total number of runs: {total_number_of_runs(filtered_ids={}, **dataframes)} \n")

print(f"Median number of runs for each attempted exercise: {quartiles_for_exercise(get_number_runs(filtered_ids={}, **dataframes))} (skewness = {skewness_for_exercises(get_number_runs(filtered_ids={}, **dataframes))}")
print(f"Median amount of time on each attempted exercise: {quartiles_for_exercise(get_time_on_exercises(filtered_ids={}, **dataframes))} (skewness = {skewness_for_exercises(get_time_on_exercises(filtered_ids={}, **dataframes))}\n")

print(f"Number of students who ended every exercise in an error state: {length_of_participants_list(filtered_ids=filter_by_students_who_ended_all_attempted_exercises_in_incorrect_state(), **dataframes)}")
print(f"Number of students who ended the exercises in a correct state:{get_end_state_of_program(filtered_ids={}, **dataframes)}\n")
print(f"Number of students who ended every exercise in a correct state: {length_of_participants_list(filtered_ids=filter_by_students_who_ended_all_attempted_exercises_in_correct_state(), **dataframes)}")
print(f"Number of exercise attempts where every run was unsuccessful: {get_number_exercises_in_error_state_for_every_run(filtered_ids={}, **dataframes)}\n")

print(f"Median edit distances between runs: {quartiles_for_exercise(get_edit_distance_between_changed_runs(filtered_ids={}, **dataframes))} (skewness = {skewness_for_exercises(get_edit_distance_between_changed_runs(filtered_ids={}, **dataframes))})")
print(f"Edit distance between first and last snapshot for each exercise: {quartiles_for_exercise(get_edit_distance_between_first_and_last_snapshot(filtered_ids={}, **dataframes))} (skewness = {skewness_for_exercises(get_edit_distance_between_first_and_last_snapshot(filtered_ids={}, **dataframes))})")
print(f"Median edit distance between first and last snapshot of each attempted exercise: {quartiles_for_exercise(get_edit_distance_between_first_and_last_snapshot(filtered_ids={}, **dataframes))} (skewness = {skewness_for_exercises(get_edit_distance_between_first_and_last_snapshot(filtered_ids={}, **dataframes))}\n")

Number of attempted exercises: [72, 71, 68, 60, 51, 322]
Total number of runs: [938, 1335, 1254, 1055, 2628, 7210] 

Median number of runs for each attempted exercise: [[1, np.float64(3.0), np.float64(7.0), np.float64(15.0), 85], [1, np.float64(6.5), np.float64(13.0), np.float64(27.5), 72], [2, np.float64(7.0), np.float64(14.0), np.float64(28.0), 81], [1, np.float64(6.0), np.float64(13.0), np.float64(21.5), 65], [2, np.float64(15.0), np.float64(35.0), np.float64(78.0), 218], [1, np.float64(6.0), np.float64(13.0), np.float64(28.75), 218]] (skewness = [np.float64(2.54153867171419), np.float64(1.3848351310496965), np.float64(1.4781479775981454), np.float64(1.3473339244569154), np.float64(1.5559481182772068), np.float64(3.3301175954270454)]
Median amount of time on each attempted exercise: [[28.961, np.float64(91.4965), np.float64(153.272), np.float64(287.08425), 1183.834], [28.72, np.float64(175.8195), np.float64(304.441), np.float64(515.3975), 1416.513], [23.403, np.float64(225.0025), np

## Statistics for the categorisation

### First move

In [30]:
first_time_runs = get_first_run_time(filtered_ids=filter_program_logs_that_ran_before_changes(), **dataframes)
print(f"Median first run time for run-first exercises: {quartiles_for_exercise(first_time_runs)[5][2]:.2f} (skewness = {skewness_for_exercises(first_time_runs)[5]:.2f})")

time_between_runs = get_time_between_runs(filtered_ids=filter_program_logs_that_ran_before_changes(), **dataframes)
print(f"Median time between runs for students who ran first: {quartiles_for_exercise(time_between_runs)[5][2]:.2f} (skewness = {skewness_for_exercises(time_between_runs)[5]:.2f})")

print(f"Number of distinct students who ran before changing in at least one of the debugging exercises: {length_of_participants_list(filtered_ids=filter_students_who_ran_before_changes(), **dataframes)}\n")

unsuccessful_after_initial_run = get_number_unsuccessful_snapshots_after_given_run(filtered_ids=filter_program_logs_that_made_changes_before_running(), run_number=1, **dataframes)
print(f"Number of change-first programs that still weren't running after initial run: {unsuccessful_after_initial_run[5]}")

first_time_change_first = get_first_run_time(filtered_ids=filter_program_logs_that_made_changes_before_running(), **dataframes)
print(f"Median first run time for change-first exercises: {quartiles_for_exercise(first_time_change_first)[5][2]:.2f} (skewness = {skewness_for_exercises(first_time_change_first)[5]}:.2f)")


Median first run time for run-first exercises: 16.81 (skewness = 4.26)
Median time between runs for students who ran first: 0.92 (skewness = 5.90)
Number of distinct students who ran before changing in at least one of the debugging exercises: 25

Number of change-first programs that still weren't running after initial run: 92
Median first run time for change-first exercises: 62.94 (skewness = 1.4657381648276149:.2f)


### Positive debugging behaviours

In [18]:
print(f"Number of students who made unsuccessful change to line 4 of exercise 4: {number_students_unsuccessful_first_change_exercise_4(**dataframes)}\n")

Number of students who made unsuccessful change to line 4 of exercise 4: 31



### Introduced additional errors

In [22]:
print(f"Number of exercise attempts who added and did not resolve a syntax error: {length_of_program_logs_list(filtered_ids=filter_students_who_added_and_did_not_resolve_syntax_errors(), **dataframes)}")

time_in_syntax_error_state = get_total_time_in_specific_error_state(filtered_ids={}, error_type='syntax', **dataframes)
print(f"Median time that students were in syntax error state for: {quartiles_for_exercise(time_in_syntax_error_state)[5][2]:.2f} (skewness = {skewness_for_exercises(time_in_syntax_error_state)[5]:.2f})\n")

print(f"Number of students who introduced at least one logical error in at least one of the exercises: {length_of_participants_list(filtered_ids=filter_students_who_introduced_logical_errors(), **dataframes)}")

Number of exercise attempts who added and did not resolve a syntax error: 75
Median time that students were in syntax error state for: 87.73 (skewness = 1.59)

Number of students who introduced at least one logical error in at least one of the exercises: 59


### Resolved errors

In [ ]:
print(f"Number of students who resolved at least one of the logical errors in at least one of the debugging exercises: {length_of_participants_list(filtered_ids=filter_students_who_resolved_logical_errors(), **dataframes)}")
print(f"Number of students who resolved at least one error by introducing a logical error: {length_of_participants_list(filtered_ids=filter_students_who_introduced_logical_error_through_resolution(), **dataframes)}\n")
print(f"Number of students who made same first change as Bukayo on exercise 1: {number_students_first_change_same_as_student_19_exercise_1(**dataframes)}\n")
print(f"Number of students who ended exercises with one or more logical errors, given that they had introduced at least one logical error:\n{get_end_state_of_program(filtered_ids=filter_students_who_made_shortcut_changes(), **dataframes)}\n")


Number of students who resolved at least one of the logical errors in at least one of the debugging exercises: 56
Number of students who resolved at least one error by introducing a logical error: 42

Number of students who made same first change as Bukayo on exercise 1: 15

Number of students who ended exercises with one or more logical errors, given that they had introduced at least one logical error:
{'Ended exercise with syntax error(s)': [3, 20, 16, 9, 7], 'Ended exercise with runtime error(s)': [11, 5, 0, 13, 0], 'Ended exercise with type error(s)': [2, 4, 2, 18, 0], 'Ended exercise with logical error(s)': [26, 20, 29, 24, 15], 'Ended exercise in correct state': [11, 16, 10, 9, 12], "Didn't attempt exercise": [1, 1, 0, 0, 0], 'Total attempts': [44, 43, 40, 35, 28]}



### Miscellaneous behaviours

In [41]:
print(f"Total number of repeated runs: {overall_total(get_number_unchanged_runs(filtered_ids={}, **dataframes))}")
print(f"- Total number of repeated unsuccessful runs: {overall_total(get_number_unchanged_unsuccessful_runs(filtered_ids={}, **dataframes))}")
print(f"- Total number of repeated successful runs: {overall_total(get_number_unchanged_successful_runs(filtered_ids={}, **dataframes))}\n")

print(f"Number of repeated successful runs by exercise number: {total_per_exercises(get_number_unchanged_successful_runs(filtered_ids={}, **dataframes))}")
print(f"Median number of unchanged runs per exercise: {quartiles_for_exercise(get_number_unchanged_runs(filtered_ids={}, **dataframes))[5]} (skewness = {skewness_for_exercises(get_number_unchanged_runs(filtered_ids={}, **dataframes))[5]:.2f})\n")

print(f"Median time between unchanged runs: {quartiles_for_exercise(get_time_between_all_unchanged_runs(filtered_ids={}, **dataframes))[5]}")
time_between_unsuccessful_unchanged_runs = get_time_between_unsuccessful_unchanged_runs(filtered_ids={}, **dataframes)
print(f"Median time between unchanged unsuccessful runs: {quartiles_for_exercise(time_between_unsuccessful_unchanged_runs)[5]} (skewness = {skewness_for_exercises(time_between_unsuccessful_unchanged_runs)[5]:.2f})\n")

#print(f"Number of students who repeatedly successfully ran their program in exercise 5: {len([x for x in get_number_unchanged_successful_runs(filtered_ids={}, **dataframes)[4] if x != 0])}\n")
#print(f"Median time between successful unchanged runs for exercise 5:  {quartiles_for_exercise(get_time_between_successful_unchanged_runs(filtered_ids={}, **dataframes))[4]} (skewness = {skewness_for_exercises(get_time_between_successful_unchanged_runs(filtered_ids={}, **dataframes))[4]})\n")

Total number of repeated runs: 4809
- Total number of repeated unsuccessful runs: 2283
- Total number of repeated successful runs: 2526

Number of repeated successful runs by exercise number: [82, 22, 121, 129, 2172, 2526]
Median number of unchanged runs per exercise: [0, np.float64(2.0), np.float64(5.0), np.float64(18.0), 213] (skewness = 4.01)

Median time between unchanged runs: [0.008, np.float64(0.193), np.float64(0.376), np.float64(0.984), 345.188]
Median time between unchanged unsuccessful runs: [0.023, np.float64(0.192), np.float64(0.367), np.float64(1.0105), 345.188] (skewness = 7.86)



## Some visualisations

## Other stats of interest
These are stats that are reported in the paper.